# 📈 Stock Candle Prediction — Upgraded
### Features added:
- ✅ You enter ANY stock ticker — not hardcoded
- ✅ Tomorrow's predicted close price shown
- ✅ Halal stock screening built-in
- ✅ Probability of UP / DOWN both shown clearly
- ✅ All previous fixes retained (normalised features, calibrated ML, regime filter, backtest)

## CELL 1 — Imports

In [167]:
import yfinance as yf
import pandas as pd
import numpy as np

from ta.trend      import EMAIndicator, MACD
from ta.momentum   import RSIIndicator
from ta.volatility import AverageTrueRange, BollingerBands

from xgboost import XGBClassifier
from sklearn.calibration   import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics         import accuracy_score, classification_report

import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded.")

✅ Libraries loaded.


## CELL 2 — Halal Stock Screener

> This cell checks whether a stock is **Halal-compliant** before allowing analysis.
> 
> **What makes a stock Haram (forbidden)?**
> - The company's primary business involves: alcohol, tobacco, pork/meat processing, conventional banking/insurance (interest-based), gambling, adult entertainment, or weapons
> 
> **What is considered Halal?**
> - Companies in: technology, real estate, FMCG (non-haram goods), healthcare, manufacturing, infrastructure, retail (non-haram), telecom
> 
> **Note:** This is a best-effort screener based on company sector/industry info from Yahoo Finance.
> Always do your own research (do your due diligence) for final confirmation.
> For a strict ruling, consult a qualified Islamic finance scholar or a certified Shariah screening service.

In [168]:
# ─── Keywords that indicate a Haram business ──────────────────────────────
HARAM_KEYWORDS = [
    # Finance (interest/riba based)
    "bank", "banking", "insurance", "financial services", "credit",
    "mortgage", "lending", "diversified financials",
    # Alcohol
    "brewery", "breweries", "distillery", "distilleries",
    "alcoholic beverages", "beverages - alcoholic", "wine", "spirits",
    # Tobacco
    "tobacco",
    # Pork / meat processing (broad)
    "pork", "swine",
    # Gambling
    "gambling", "casino", "gaming",
    # Adult
    "adult entertainment",
    # Weapons
    "defense", "defence", "weapons", "arms",
]

# ─── Known halal-compliant NSE stocks (community-verified list) ────────────
# This is a starter list. You can add more tickers you have personally verified.
KNOWN_HALAL_STOCKS = {
    # Technology
    "TCS.NS", "INFY.NS", "WIPRO.NS", "HCLTECH.NS", "TECHM.NS",
    "MPHASIS.NS", "COFORGE.NS", "PERSISTENT.NS", "LTIM.NS",
    # Telecom
    "BHARTIARTL.NS",
    # FMCG / Consumer
    "HINDUNILVR.NS", "DABUR.NS", "MARICO.NS", "COLPAL.NS",
    "BRITANNIA.NS", "NESTLEIND.NS",
    # Retail / E-commerce
    "DMART.NS", "TRENT.NS",
    # Healthcare / Pharma
    "SUNPHARMA.NS", "DRREDDY.NS", "CIPLA.NS", "DIVISLAB.NS",
    "APOLLOHOSP.NS", "FORTIS.NS",
    # Cement / Infrastructure
    "ULTRACEMCO.NS", "AMBUJACEM.NS", "ACC.NS", "SHREECEM.NS",
    # Auto / Manufacturing
    "MARUTI.NS", "TATAMOTORS.NS", "M&M.NS", "BAJAJ-AUTO.NS",
    "HEROMOTOCO.NS", "EICHERMOT.NS",
    # Metals / Commodities
    "TATASTEEL.NS", "JSWSTEEL.NS", "HINDALCO.NS",
    # Real estate
    "DLF.NS", "GODREJPROP.NS", "ANANTRAJ.NS",
    # Energy (non-interest)
    "RELIANCE.NS", "ONGC.NS", "NTPC.NS", "POWERGRID.NS",
    # Others
    "ADANIENT.NS", "ADANIPORTS.NS", "ADANIGREEN.NS",
    "TITAN.NS", "PIDILITIND.NS", "ASIANPAINT.NS",
}


def check_halal(ticker: str) -> dict:
    """
    Checks if a stock is likely Halal-compliant.
    Returns a dict with: is_halal (bool), confidence, reason, sector, industry
    """
    ticker_upper = ticker.upper()

    # Step 1: Check known list first (most reliable)
    if ticker_upper in KNOWN_HALAL_STOCKS:
        return {
            "is_halal"   : True,
            "confidence" : "High",
            "reason"     : "Stock is in the community-verified Halal list.",
            "sector"     : "—",
            "industry"   : "—",
        }

    # Step 2: Fetch company info from Yahoo Finance
    try:
        info     = yf.Ticker(ticker).info
        sector   = str(info.get("sector",   "")).lower()
        industry = str(info.get("industry", "")).lower()
        name     = str(info.get("longName", ticker))
    except Exception:
        return {
            "is_halal"   : None,
            "confidence" : "Unknown",
            "reason"     : "Could not fetch company info. Check the ticker symbol.",
            "sector"     : "—",
            "industry"   : "—",
        }

    # Step 3: Scan sector and industry for haram keywords
    combined = sector + " " + industry
    flagged  = [kw for kw in HARAM_KEYWORDS if kw in combined]

    if flagged:
        return {
            "is_halal"   : False,
            "confidence" : "High",
            "reason"     : f"Flagged as potentially Haram. Keywords found: {flagged}",
            "sector"     : sector.title(),
            "industry"   : industry.title(),
        }
    else:
        return {
            "is_halal"   : True,
            "confidence" : "Medium",
            "reason"     : "No Haram keywords detected in sector/industry. Always verify manually.",
            "sector"     : sector.title(),
            "industry"   : industry.title(),
        }


print("✅ Halal screener ready.")

✅ Halal screener ready.


## CELL 3 — 🔧 Enter Your Stock Here

> **This is the only cell you need to edit.**
> Change the `STOCK_TICKER` to any NSE stock you want to analyse.
> Format: `TICKERSYMBOL.NS` for NSE India stocks.
> Examples: `DMART.NS`, `INFY.NS`, `RELIANCE.NS`, `TCS.NS`

In [169]:
# ══════════════════════════════════════════════════
#  ✏️  CHANGE THIS to the stock you want to analyse
# ══════════════════════════════════════════════════
STOCK_TICKER = "HEROMOTOCO.NS"

# How many years of history to train on (2–5 recommended)
PERIOD = "4y"
# ══════════════════════════════════════════════════


# ─── Halal Check ──────────────────────────────────
print(f"\n{'═'*50}")
print(f"   HALAL SCREENING: {STOCK_TICKER}")
print(f"{'═'*50}")

halal_result = check_halal(STOCK_TICKER)

print(f"  Sector       : {halal_result['sector']}")
print(f"  Industry     : {halal_result['industry']}")
print(f"  Confidence   : {halal_result['confidence']}")
print(f"  Reason       : {halal_result['reason']}")

if halal_result["is_halal"] is False:
    print(f"\n  ❌  HARAM — This stock is NOT Halal-compliant.")
    print(f"      Analysis stopped. Please choose a different stock.")
    print(f"{'═'*50}\n")
    raise SystemExit("Haram stock detected. Change STOCK_TICKER and re-run.")

elif halal_result["is_halal"] is None:
    print(f"\n  ⚠️   UNKNOWN — Could not verify. Proceeding with caution.")
    print(f"      Please verify this stock manually before trading.")

else:
    print(f"\n  ✅  HALAL — This stock appears to be Halal-compliant.")

print(f"{'═'*50}\n")


══════════════════════════════════════════════════
   HALAL SCREENING: HEROMOTOCO.NS
══════════════════════════════════════════════════
  Sector       : —
  Industry     : —
  Confidence   : High
  Reason       : Stock is in the community-verified Halal list.

  ✅  HALAL — This stock appears to be Halal-compliant.
══════════════════════════════════════════════════



## CELL 4 — Download Data

In [170]:
data = yf.download(STOCK_TICKER, period=PERIOD)
data.index   = pd.to_datetime(data.index)
data.columns = data.columns.get_level_values(0)

if len(data) < 200:
    raise ValueError(f"Not enough data ({len(data)} rows). Need at least 200. Try a longer PERIOD.")

print(f"Stock        : {STOCK_TICKER}")
print(f"Total rows   : {len(data)}")
print(f"Date range   : {data.index[0].date()} → {data.index[-1].date()}")
data.tail(3)

[*********************100%***********************]  1 of 1 completed

Stock        : HEROMOTOCO.NS
Total rows   : 990
Date range   : 2022-06-06 → 2026-06-05


Price,Close,High,Low,Open,Volume
Date,,,,,
2026-06-03,4840.899902,4873.799805,4815.000000,4856.000000,571637
2026-06-04,4882.299805,4980.000000,4850.399902,4850.399902,783946
2026-06-05,4838.200195,4904.500000,4791.100098,4898.000000,324101


## CELL 5 — Indicators & Features

In [171]:
close      = data["Close"].squeeze()
high       = data["High"].squeeze()
low        = data["Low"].squeeze()
open_price = data["Open"].squeeze()

# ── Trend ─────────────────────────────────────────────────────────────────
data["EMA"] = EMAIndicator(close=close, window=50).ema_indicator()

# ── Momentum ──────────────────────────────────────────────────────────────
data["RSI"] = RSIIndicator(close=close, window=14).rsi()

# ── Volatility ────────────────────────────────────────────────────────────
data["ATR"]    = AverageTrueRange(high=high, low=low, close=close, window=14).average_true_range()
bb             = BollingerBands(close=close)
data["BB_high"] = bb.bollinger_hband()
data["BB_low"]  = bb.bollinger_lband()

# ── MACD ──────────────────────────────────────────────────────────────────
macd_ind            = MACD(close=close)
data["MACD"]        = macd_ind.macd()
data["MACD_signal"] = macd_ind.macd_signal()

# ── Candle anatomy ────────────────────────────────────────────────────────
data["Body"]       = abs(close - open_price)
data["Upper_Wick"] = high  - np.maximum(close, open_price)
data["Lower_Wick"] = np.minimum(close, open_price) - low
data["Body_Ratio"] = data["Body"] / (high - low + 1e-9)

# ── Returns & Volume ──────────────────────────────────────────────────────
data["Return"]      = close.pct_change()
data["Return_prev"] = data["Return"].shift(1)
data["High_Low"]    = (high - low) / close

# ── Normalised features (no price-level leakage) ──────────────────────────
data["EMA_dist"]  = (close - data["EMA"]) / close
bb_range          = (data["BB_high"] - data["BB_low"]).replace(0, np.nan)
data["BB_pos"]    = (close - data["BB_low"]) / bb_range
data["MACD_hist"] = data["MACD"] - data["MACD_signal"]
data["ATR_pct"]   = data["ATR"] / close
data["RSI_norm"]  = data["RSI"] / 100
data["RSI_slope"] = data["RSI"].diff(3) / 100
data["Vol_ratio"] = data["Volume"] / data["Volume"].rolling(20).mean()

min_body             = 0.001 * close
body_safe            = data["Body"].clip(lower=min_body)
data["Wick_ratio"]   = data["Lower_Wick"] / body_safe
data["Upper_ratio"]  = data["Upper_Wick"] / body_safe

# ── Candle patterns ───────────────────────────────────────────────────────
min_body_mask = data["Body"] > (0.001 * close)

data["Bullish_Engulfing"] = (
    (data["Close"].shift(1) < data["Open"].shift(1)) &
    (data["Close"] > data["Open"]) &
    (data["Close"] > data["Open"].shift(1)) &
    (data["Open"]  < data["Close"].shift(1)) &
    min_body_mask
)
data["Hammer"] = (
    (data["Lower_Wick"] > 2 * data["Body"]) &
    (data["Upper_Wick"] < data["Body"]) &
    min_body_mask
)
data["Shooting_Star"] = (
    (data["Upper_Wick"] > 2 * data["Body"]) &
    (data["Lower_Wick"] < data["Body"]) &
    min_body_mask
)

print("✅ All indicators and features computed.")

✅ All indicators and features computed.


## CELL 6 — Label & Clean

In [172]:
data["Tomorrow_Close"] = close.shift(-1)
data["Label"]          = (data["Tomorrow_Close"] > close).astype(int)

data.replace([np.inf, -np.inf], np.nan, inplace=True)

FEATURE_COLS = [
    "RSI_norm", "RSI_slope", "EMA_dist", "BB_pos", "MACD_hist",
    "ATR_pct", "Body_Ratio", "Wick_ratio", "Upper_ratio",
    "Return", "Return_prev", "Vol_ratio", "High_Low"
]

data.dropna(subset=FEATURE_COLS + ["Label"], inplace=True)

counts = data["Label"].value_counts()
print(f"Label balance — UP: {counts[1]} ({counts[1]/len(data):.1%})  DOWN: {counts[0]} ({counts[0]/len(data):.1%})")
print(f"Total usable rows: {len(data)}")

Label balance — UP: 487 (51.8%)  DOWN: 454 (48.2%)
Total usable rows: 941


## CELL 7 — Train / Test Split

In [173]:
X = data[FEATURE_COLS]
y = data["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
spw   = round(n_neg / n_pos, 2)

print(f"Train: {len(X_train)} rows  |  Test: {len(X_test)} rows")
print(f"scale_pos_weight: {spw}")

Train: 752 rows  |  Test: 189 rows
scale_pos_weight: 0.91


## CELL 8 — Train XGBoost + Calibrate

In [174]:
base_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=spw,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)

model = CalibratedClassifierCV(base_model, cv=5, method='isotonic')
model.fit(X_train, y_train)

preds    = model.predict(X_test)
accuracy = accuracy_score(y_test, preds)

print(f"✅ Model trained.")
print(f"Test Accuracy : {accuracy:.2%}")
print()
print(classification_report(y_test, preds, target_names=["DOWN", "UP"]))

✅ Model trained.
Test Accuracy : 45.50%

              precision    recall  f1-score   support

        DOWN       0.42      0.23      0.30        95
          UP       0.47      0.68      0.55        94

    accuracy                           0.46       189
   macro avg       0.45      0.46      0.43       189
weighted avg       0.44      0.46      0.43       189



## CELL 9 — Signal Generator Function

In [175]:
def generate_signal(row, model, feature_cols):
    x = row[feature_cols].values.reshape(1, -1)

    # Layer 1: ML calibrated probability
    prob_up   = model.predict_proba(x)[0][1]
    prob_down = 1 - prob_up

    # Layer 2: Candle pattern boost/penalty
    boost   = 0.0
    reasons = []
    if row.get("Bullish_Engulfing", False):
        boost += 0.08
        reasons.append("+Bullish Engulfing")
    if row.get("Hammer", False):
        boost += 0.05
        reasons.append("+Hammer")
    if row.get("Shooting_Star", False):
        boost -= 0.10
        reasons.append("-Shooting Star")

    # Layer 3: Regime filter
    regime_ok = (
        row["RSI_norm"]       < 0.75  and
        row["ATR_pct"]        > 0.005 and
        abs(row["EMA_dist"]) < 0.15
    )

    final_score = float(np.clip((prob_up + boost) * 100, 0, 100))

    # Recommendation
    if not regime_ok:
        rec = "Skip — bad regime"
    elif final_score >= 72:
        rec = "Strong Buy"
    elif final_score >= 60:
        rec = "Buy"
    elif final_score <= 35:
        rec = "Avoid"
    else:
        rec = "Neutral"

    return {
        "prob_up"    : prob_up,
        "prob_down"  : prob_down,
        "boost"      : boost,
        "final_score": final_score,
        "regime_ok"  : regime_ok,
        "reasons"    : reasons if reasons else ["No pattern"],
        "rec"        : rec,
    }

print("✅ Signal function defined.")

✅ Signal function defined.


## CELL 10 — Tomorrow's Predicted Close Price

> **How does this work?**
> 
> The ML model predicts direction (UP or DOWN) with a probability.
> To estimate tomorrow's close price, we use the historical average % move
> on days when the model was similarly confident — then apply that to today's close.
> 
> This is an **estimate**, not a guarantee. Markets are unpredictable.
> Use it as a reference range, not an exact target.

In [176]:
def predict_tomorrow_price(data, model, feature_cols, confidence_threshold=0.55):
    """
    Estimates tomorrow's close price using:
    1. The ML model's UP/DOWN probability
    2. Historical average move on similar-confidence days
    3. ATR-based range (best case / worst case)
    """
    # Today's row (latest available candle)
    today       = data.iloc[-1]
    today_close = float(today["Close"])
    atr         = float(today["ATR"])

    # Get today's ML probability
    x         = today[feature_cols].values.reshape(1, -1)
    prob_up   = model.predict_proba(x)[0][1]
    prob_down = 1 - prob_up

    # ── Historical average move on training data ───────────────────────────
    train_data = data.iloc[:int(len(data) * 0.8)].copy()
    train_probs = model.predict_proba(train_data[feature_cols].values)[:, 1]
    train_data["ML_prob"] = train_probs

    # Find days where model was similarly confident (within ±10%)
    similar_days = train_data[
        (train_data["ML_prob"] >= prob_up - 0.10) &
        (train_data["ML_prob"] <= prob_up + 0.10) &
        (train_data["Tomorrow_Close"].notna())
    ].copy()

    if len(similar_days) >= 5:
        similar_days["actual_move_pct"] = (
            (similar_days["Tomorrow_Close"] - similar_days["Close"]) / similar_days["Close"]
        )
        avg_move_pct = similar_days["actual_move_pct"].mean()
    else:
        # Fallback: use overall average up/down move
        avg_up_move   = train_data[train_data["Return"] > 0]["Return"].mean()
        avg_down_move = train_data[train_data["Return"] < 0]["Return"].mean()
        avg_move_pct  = (prob_up * avg_up_move) + (prob_down * avg_down_move)

    # ── Price estimates ────────────────────────────────────────────────────
    predicted_close  = today_close * (1 + avg_move_pct)
    bullish_estimate = today_close + atr        # optimistic: move up 1 ATR
    bearish_estimate = today_close - atr        # pessimistic: move down 1 ATR

    return {
        "today_close"     : today_close,
        "predicted_close" : predicted_close,
        "bullish_estimate": bullish_estimate,
        "bearish_estimate": bearish_estimate,
        "avg_move_pct"    : avg_move_pct,
        "prob_up"         : prob_up,
        "prob_down"       : prob_down,
        "atr"             : atr,
        "similar_days"    : len(similar_days),
    }

print("✅ Price prediction function defined.")

✅ Price prediction function defined.


## CELL 11 — Backtest

In [177]:
test_data = data.iloc[len(X_train):].copy()

test_probs = model.predict_proba(X_test)[:, 1]
test_data["ML_prob"] = test_probs

test_data["Candle_boost"] = (
    test_data["Bullish_Engulfing"].astype(float) * 0.08 +
    test_data["Hammer"].astype(float)            * 0.05 +
    test_data["Shooting_Star"].astype(float)     * -0.10
)
test_data["Final_score"] = np.clip(
    (test_data["ML_prob"] + test_data["Candle_boost"]) * 100, 0, 100
)
test_data["Regime_OK"] = (
    (test_data["RSI_norm"]       < 0.75) &
    (test_data["ATR_pct"]        > 0.005) &
    (test_data["EMA_dist"].abs() < 0.15)
)

buy_signals = test_data[(test_data["Final_score"] >= 60) & test_data["Regime_OK"]]
hit_rate    = (buy_signals["Label"] == 1).mean() if len(buy_signals) > 0 else 0
baseline    = test_data["Label"].mean()

print("=== Backtest Results ===")
print(f"Total test rows          : {len(test_data)}")
print(f"BUY signals fired        : {len(buy_signals)}")
print(f"Signal hit rate          : {hit_rate:.1%}")
print(f"Baseline (always UP)     : {baseline:.1%}")
print(f"Edge over baseline       : {(hit_rate - baseline)*100:+.1f} pp")
print()
if hit_rate > baseline:
    print("✅ Signal BEATS baseline — useful edge found.")
else:
    print("⚠️  Signal does not beat baseline — consider tuning thresholds.")

=== Backtest Results ===
Total test rows          : 189
BUY signals fired        : 45
Signal hit rate          : 51.1%
Baseline (always UP)     : 49.7%
Edge over baseline       : +1.4 pp

✅ Signal BEATS baseline — useful edge found.


## CELL 12 — 🎯 Final Output: Full Signal + Tomorrow's Price

In [189]:
latest       = data.iloc[-1]
signal       = generate_signal(latest, model, FEATURE_COLS)
price_pred   = predict_tomorrow_price(data, model, FEATURE_COLS)

today_date   = data.index[-1].date()
prob_up_pct  = signal["prob_up"]   * 100
prob_dn_pct  = signal["prob_down"] * 100
score        = signal["final_score"]
rec          = signal["rec"]

# ── Emoji helpers ──────────────────────────────────────────────────────────
rec_emoji = {
    "Strong Buy"       : "🟢",
    "Buy"              : "🟢",
    "Neutral"          : "🟡",
    "Avoid"            : "🔴",
    "Skip — bad regime": "⛔",
}.get(rec, "⚪")

up_bar   = "█" * int(prob_up_pct / 5)
down_bar = "█" * int(prob_dn_pct / 5)

print(f"""
╔══════════════════════════════════════════════════════╗
║   STOCK SIGNAL REPORT — {STOCK_TICKER:<15}{today_date}    ║
╠══════════════════════════════════════════════════════╣
║                                                      ║
║  TODAY'S CLOSE     :  ₹{latest['Close']:>10.2f}                    ║
║  RSI               :  {latest['RSI']:>6.1f}  {'(Overbought ⚠️)' if latest['RSI'] > 70 else '(Oversold ✅)' if latest['RSI'] < 30 else '(Normal ✅)':20} ║
║  EMA Distance      :  {latest['EMA_dist']*100:>+6.1f}%                        ║
║  BB Position       :  {latest['BB_pos']*100:>6.1f}%                        ║
║  ATR (volatility)  :  ₹{latest['ATR']:>8.2f}  ({latest['ATR_pct']*100:.2f}% of price)     ║
║                                                      ║
╠══════════════════════════════════════════════════════╣
║  PROBABILITY                                         ║
║                                                      ║
║  UP   {prob_up_pct:>5.1f}%  {up_bar:<20}                ║
║  DOWN {prob_dn_pct:>5.1f}%  {down_bar:<20}                ║
║                                                      ║
║  ML Probability    :  {prob_up_pct:>5.1f}% UP                    ║
║  Candle Boost      :  {signal['boost']*100:>+5.1f}%  {str(signal['reasons']):<25}  ║
║  Final Score       :  {score:>5.1f} / 100                     ║
║  Regime OK         :  {'✅ YES' if signal['regime_ok'] else '❌ NO (skip trading today)':30}  ║
║                                                      ║
╠══════════════════════════════════════════════════════╣
║  TOMORROW'S PREDICTED CLOSE                          ║
║                                                      ║
║  Best estimate     :  ₹{price_pred['predicted_close']:>10.2f}                  ║
║  Bullish scenario  :  ₹{price_pred['bullish_estimate']:>10.2f}  (up ~1 ATR)        ║
║  Bearish scenario  :  ₹{price_pred['bearish_estimate']:>10.2f}  (down ~1 ATR)      ║
║  Expected move     :  {price_pred['avg_move_pct']*100:>+6.2f}%                          ║
║  (Based on {price_pred['similar_days']:>3} similar historical days)            ║
║                                                      ║
╠══════════════════════════════════════════════════════╣
║                                                      ║
║  {rec_emoji}  RECOMMENDATION :  {rec:<33}  ║
║                                                      ║
║  ☪️  HALAL STATUS  :  {('✅ Verified' if halal_result['is_halal'] else '⚠️ Unverified'):30}  ║
║                                                      ║
╚══════════════════════════════════════════════════════╝
""")

print("⚠️  DISCLAIMER: This is an educational prediction tool.")
print("   Always do your own research before making any trade.")
print("   Past signals do not guarantee future results.")

# ── FUTURE AUTO-TRADE HOOK ─────────────────────────────────────────────────
# from kiteconnect import KiteConnect
# kite = KiteConnect(api_key="YOUR_API_KEY")
# kite.set_access_token("YOUR_ACCESS_TOKEN")
#
# ticker_clean = STOCK_TICKER.replace(".NS", "")
# if signal["rec"] in ["Strong Buy", "Buy"] and signal["regime_ok"] and halal_result["is_halal"]:
#     kite.place_order(
#         tradingsymbol=ticker_clean,
#         exchange=kite.EXCHANGE_NSE,
#         transaction_type=kite.TRANSACTION_TYPE_BUY,
#         quantity=10,
#         order_type=kite.ORDER_TYPE_MARKET,
#         product=kite.PRODUCT_CNC
#     )
#     print(f"Order placed for {ticker_clean}!")


╔══════════════════════════════════════════════════════╗
║   STOCK SIGNAL REPORT — HEROMOTOCO.NS  2026-06-05    ║
╠══════════════════════════════════════════════════════╣
║                                                      ║
║  TODAY'S CLOSE     :  ₹   4838.20                    ║
║  RSI               :    39.3  (Normal ✅)           ║
║  EMA Distance      :    -5.4%                        ║
║  BB Position       :    14.8%                        ║
║  ATR (volatility)  :  ₹  134.67  (2.78% of price)     ║
║                                                      ║
╠══════════════════════════════════════════════════════╣
║  PROBABILITY                                         ║
║                                                      ║
║  UP    46.0%  █████████                           ║
║  DOWN  54.0%  ██████████                          ║
║                                                      ║
║  ML Probability    :   46.0% UP                    ║
║  Candle Boost      :   +0.0%  ['No pa